# Phase I — CUB train-only selection

This notebook audits the locked CUB split, extracts **training features only**, and runs fixed-transfer plus equal-budget train-validation selection. It never creates `test.pt` and never evaluates the held-out CUB test split. Run all cells in order, then return the ZIP for review.

In [ ]:
# === Edit paths only. Do not edit model/search/gate values. ===
REPO_GIT_URL = 'https://github.com/ZaPhat206/SOHO-CL.git'
REPO_BRANCH = 'feature/crt-soho'
WORK_DIR = '/content/SOHO-CL'
DRIVE_ROOT = '/content/drive/MyDrive/T-SOHO'
DRIVE_TRAIN_CACHE = f'{DRIVE_ROOT}/cub_train_feature_cache'
TRAIN_CACHE_DIR = '/content/cub_train_feature_cache'
GATE_CACHE_DIR = f'{DRIVE_ROOT}/phasei_cub_gate_cache_v2_float64'
OUTPUT_DIR = f'{DRIVE_ROOT}/phasei_cub_train_only_selection_v2_float64'
CHECKPOINT_SOURCE = 'huggingface'  # or 'google_drive'
DRIVE_CHECKPOINT_PATH = f'{DRIVE_ROOT}/model.safetensors'
BATCH_SIZE = 128
CHECKPOINT_SIZE = 346284714
CHECKPOINT_SHA256 = '32aa17d6e17b43500f531d5f6dc9bc93e56ed8841b8a75682e1bb295d722405b'
MANIFEST_SHA256 = 'e234b97080442c113578c7d477a2eecfc60ad2a9484fbea42ea1720fd9dd62d9'


In [ ]:
# Runtime setup. Safe after a previous disconnected run.
from google.colab import drive
drive.mount('/content/drive')
import hashlib, json, os, shutil, subprocess, sys, time
from pathlib import Path
import torch
assert torch.cuda.is_available(), 'Select Runtime > Change runtime type > T4 GPU.'
os.chdir('/content')
shutil.rmtree(WORK_DIR, ignore_errors=True)
subprocess.run(['git', 'clone', '--branch', REPO_BRANCH, REPO_GIT_URL, WORK_DIR], check=True)
os.chdir(WORK_DIR)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements-kaggle.txt', 'kagglehub', 'huggingface_hub'], check=True)
commit = subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip()
print('repo commit:', commit)
print('GPU:', torch.cuda.get_device_name(0))


In [ ]:
# Obtain and verify the exact frozen ViT checkpoint.
if CHECKPOINT_SOURCE == 'huggingface':
    from huggingface_hub import hf_hub_download
    CHECKPOINT_PATH = hf_hub_download('timm/vit_base_patch16_224.augreg2_in21k_ft_in1k', 'model.safetensors')
elif CHECKPOINT_SOURCE == 'google_drive':
    CHECKPOINT_PATH = DRIVE_CHECKPOINT_PATH
else:
    raise ValueError('CHECKPOINT_SOURCE must be huggingface or google_drive')
checkpoint = Path(CHECKPOINT_PATH)
digest = hashlib.sha256(checkpoint.read_bytes()).hexdigest()
assert checkpoint.stat().st_size == CHECKPOINT_SIZE
assert digest == CHECKPOINT_SHA256
print('checkpoint PASS:', checkpoint, digest)


In [ ]:
# Download and content-audit the exact processed CUB split (progress every 1,000 images).
import kagglehub
DATASET_PATH = kagglehub.dataset_download('zaphat206/cub-200-2011')
DATASET_AUDIT = '/content/cub_dataset_audit.json'
audit_command = [sys.executable, '-u', 'tools/cub_dataset_audit.py', '--root', DATASET_PATH, '--output', DATASET_AUDIT, '--expected-identity-sha256', 'e374af9b576cb6b3503198ef3ea30fd0aa9d2e18c230ff8064e21d4f644af2ca', '--progress-every', '1000']
subprocess.run(audit_command, check=True)
audit = json.loads(Path(DATASET_AUDIT).read_text())
print('CUB identity PASS:', audit['dataset_identity_sha256'])
print('train/test:', audit['train']['image_count'], audit['test']['image_count'])


In [ ]:
# Correctness tests. No CUB test feature is opened.
subprocess.run([sys.executable, '-m', 'pytest', '-q', 'tests/test_cub_dataset_audit.py', 'tests/test_cub_data_utils.py', 'tests/test_phasei_cub_selection.py', 'tests/test_crt_gate_runner.py'], check=True)
print('Phase I correctness gate: PASS')


In [ ]:
# Restore or extract TRAIN embeddings only. One progress line is printed per task.
local_cache = Path(TRAIN_CACHE_DIR)
drive_cache = Path(DRIVE_TRAIN_CACHE)
if not (local_cache / 'metadata.json').is_file():
    if (drive_cache / 'metadata.json').is_file():
        shutil.copytree(drive_cache, local_cache, dirs_exist_ok=True)
        print('Restored train-only feature cache from Drive.')
    else:
        command = [sys.executable, '-u', 'tools/experiment_runner.py', '--extract-features-only', '--extract-train-only', '--root', DATASET_PATH, '--backbone-checkpoint', CHECKPOINT_PATH, '--backbone-checkpoint-size', str(CHECKPOINT_SIZE), '--backbone-checkpoint-sha256', CHECKPOINT_SHA256, '--feature-cache-dir', TRAIN_CACHE_DIR, '--output-dir', '/content/phasei_extract_only', '--dataset', 'CUB-200-2011', '--model-name', 'vit_base_patch16_224', '--data-augmentation', 'vit', '--seed', '1993', '--num-classes', '200', '--num-tasks', '20', '--device', 'cuda', '--batch-size', str(BATCH_SIZE), '--num-workers', '2']
        print('Starting 20-task TRAIN-only ViT extraction...', flush=True)
        subprocess.run(command, check=True)
        assert not (local_cache / 'test.pt').exists()
        if drive_cache.exists():
            raise RuntimeError('Incomplete/invalid Drive cache exists; inspect it instead of overwriting.')
        shutil.copytree(local_cache, drive_cache)
        print('Saved validated train-only cache to Drive.')
metadata = json.loads((local_cache / 'metadata.json').read_text())
assert metadata['test_features_materialized'] is False
assert not (local_cache / 'test.pt').exists()
print('train cache PASS:', metadata['train_shape'], 'test.pt absent')


In [ ]:
# Locked train-only selection. Every candidate is cached; rerun safely after interruption.
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
shutil.copy2(DATASET_AUDIT, Path(OUTPUT_DIR) / 'cub_dataset_audit.json')
shutil.copy2('configs/phasei_cub_train_only_selection.json', Path(OUTPUT_DIR) / 'locked_manifest.json')
command = [sys.executable, '-u', 'tools/phasei_cub_selection.py', '--manifest', 'configs/phasei_cub_train_only_selection.json', '--manifest-sha256', MANIFEST_SHA256, '--dataset-audit', DATASET_AUDIT, '--feature-cache-dir', TRAIN_CACHE_DIR, '--gate-cache-dir', GATE_CACHE_DIR, '--output-dir', OUTPUT_DIR, '--device', 'cuda']
log_path = Path(OUTPUT_DIR) / 'selection.log'
print('Starting/resuming locked CUB train-only selection (125 grid candidates).', flush=True)
with log_path.open('a', encoding='utf-8') as log:
    process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in process.stdout:
        print(line, end='', flush=True)
        log.write(line); log.flush()
    returncode = process.wait()
assert returncode == 0, 'Selection runner failed; return the traceback without editing the grid.'
assert (Path(OUTPUT_DIR) / 'selection_results.json').is_file()
print('Phase I selection process: COMPLETE')


In [ ]:
# Show compact validation result and download evidence. STOP after this cell.
result = json.loads((Path(OUTPUT_DIR) / 'selection_results.json').read_text())
print('status:', result['status'])
print('held_out_test_authorized:', result['held_out_test_authorized'])
for method, item in result['selected_equal_budget'].items():
    print(f"{method:24s} val_AA={item['validation_average_incremental_accuracy']:.4f} config={item['candidate_config']}")
print('gates:', json.dumps(result['gates'], indent=2))
archive = shutil.make_archive('/content/phasei_cub_train_only_selection', 'zip', root_dir=OUTPUT_DIR)
print('artifact SHA-256:', hashlib.sha256(Path(archive).read_bytes()).hexdigest())
from google.colab import files
files.download(archive)
print('STOP. Return the ZIP for audit; do not evaluate CUB test yet.')
